In [1]:
import pandas as pd
import numpy as np

orders = pd.read_csv("../data/processed/orders_clean.csv")
order_items = pd.read_csv("../data/processed/order_items_clean.csv")
products = pd.read_csv("../data/processed/products_clean.csv")

print("Orders shape:", orders.shape)
print("Order Items shape:", order_items.shape)
print("Products shape:", products.shape)

Orders shape: (99441, 12)
Order Items shape: (112650, 8)
Products shape: (32951, 9)


In [2]:
# Create Customer–Product Interaction Data
customer_product = orders[
    ["order_id", "customer_id"]
].merge(
    order_items[
        ["order_id", "product_id"]
    ],
    on = "order_id",
    how = "inner"
)
print("Customer-Product shape:", customer_product.shape)
print("\nColumns:")
print(customer_product.columns.tolist())

print("\nSample:")
print(customer_product.head())

Customer-Product shape: (112650, 3)

Columns:
['order_id', 'customer_id', 'product_id']

Sample:
                           order_id                       customer_id  \
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
1  53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef   
2  47770eb9100c2d0c44946d9cf07ec65d  41ce2a54c0b03bf3443c3d931a367089   
3  949d5b44dbf5de918fe9c16f97b45f8a  f88197465ea7920adcdbec7375364d82   
4  ad21c59c0840e6cb83a9ceb5573f8159  8ab97904e6daea8866dbdbc4fb7aad2c   

                         product_id  
0  87285b34884572647811a353c7ac498a  
1  595fac2a385ac33a80bd5114aec74eb8  
2  aa4383b373c6aca5d8797843e5594415  
3  d0b61bfb1de832b15ba9d266ca96e5b0  
4  65266b2da20d04dbe00c5c2d3bb7859e  


In [3]:
# Remove duplicate customer-product interactions
customer_product_unique = customer_product[
    ["customer_id", "product_id"]
].drop_duplicates()

print("Original interactions:", len(customer_product))
print("Unique interactions:", len(customer_product_unique))

print("\nDuplicate interactions removed:",
      len(customer_product) - len(customer_product_unique))

print("\nSample:")
print(customer_product_unique.head())


Original interactions: 112650
Unique interactions: 102425

Duplicate interactions removed: 10225

Sample:
                        customer_id                        product_id
0  9ef432eb6251297304e76186b10a928d  87285b34884572647811a353c7ac498a
1  b0830fb4747a6c6d20dea0b8c802d7ef  595fac2a385ac33a80bd5114aec74eb8
2  41ce2a54c0b03bf3443c3d931a367089  aa4383b373c6aca5d8797843e5594415
3  f88197465ea7920adcdbec7375364d82  d0b61bfb1de832b15ba9d266ca96e5b0
4  8ab97904e6daea8866dbdbc4fb7aad2c  65266b2da20d04dbe00c5c2d3bb7859e


In [5]:
purchase_frequency = (
    customer_product
    .groupby(["customer_id", "product_id"])
    .size()
    .reset_index(name="purchase_count")
)

print("Purchase frequency shape:", purchase_frequency.shape)

print("\nSample:")
print(purchase_frequency.head())

print("\nPurchase count distribution:")
print(purchase_frequency["purchase_count"].value_counts().sort_index())

Purchase frequency shape: (102425, 3)

Sample:
                        customer_id                        product_id  \
0  00012a2ce6f8dcda20d059ce98491703  64315bd8c0c47303179dd2e25b579d00   
1  000161a058600d5901f007fab4c27140  84183944dc7cddca87a5d384452c1d3c   
2  0001fd6190edaaf884bcaf3d49edf079  9df2b21ec85378d71df4404712e17478   
3  0002414f95344307404f0ace7a26f1d5  af3ec22cce878225aae6d9eb6c7a78eb   
4  000379cdec625522490c315e70c7a9fb  868b3136c5b206f91b8208fbfdf2cb7c   

   purchase_count  
0               1  
1               1  
2               1  
3               1  
4               1  

Purchase count distribution:
purchase_count
1     95337
2      5382
3       953
4       390
5       168
6       172
7         4
8         2
9         2
10        5
11        1
12        2
13        1
14        2
15        2
20        2
Name: count, dtype: int64


In [6]:
product_info = products[
    ["product_id", "product_category_name"]
].drop_duplicates()

purchase_data = purchase_frequency.merge(
    product_info,
    on="product_id",
    how="left"
)

print("Purchase data shape:", purchase_data.shape)

print("\nColumns:")
print(purchase_data.columns.tolist())

print("\nSample:")
print(purchase_data.head())

print("\nMissing product categories:")
print(purchase_data["product_category_name"].isna().sum())

Purchase data shape: (102425, 4)

Columns:
['customer_id', 'product_id', 'purchase_count', 'product_category_name']

Sample:
                        customer_id                        product_id  \
0  00012a2ce6f8dcda20d059ce98491703  64315bd8c0c47303179dd2e25b579d00   
1  000161a058600d5901f007fab4c27140  84183944dc7cddca87a5d384452c1d3c   
2  0001fd6190edaaf884bcaf3d49edf079  9df2b21ec85378d71df4404712e17478   
3  0002414f95344307404f0ace7a26f1d5  af3ec22cce878225aae6d9eb6c7a78eb   
4  000379cdec625522490c315e70c7a9fb  868b3136c5b206f91b8208fbfdf2cb7c   

   purchase_count product_category_name  
0               1            brinquedos  
1               1          beleza_saude  
2               1                 bebes  
3               1            cool_stuff  
4               1       cama_mesa_banho  

Missing product categories:
0


In [7]:
# Find the most purchased products
popular_products = (
    purchase_data
    .groupby(
        ["product_id", "product_category_name"]
    )["purchase_count"]
    .sum()
    .reset_index()
    .sort_values(
        "purchase_count",
        ascending=False
    )
)

print("Top 10 most purchased products:")
print(popular_products.head(10))

Top 10 most purchased products:
                             product_id   product_category_name  \
22112  aca2eb7d00ea1a7b8ebd4e68314663af        moveis_decoracao   
19742  99a4788cb24856965c36a24e339b6058         cama_mesa_banho   
8613   422879e10f46682990de24d770e7f83d      ferramentas_jardim   
7364   389d119b48cf3043d311335e499d9c6b      ferramentas_jardim   
7079   368c6c730842d78016ad823897a372db      ferramentas_jardim   
10840  53759a2ecddad2bb87a079a1f1519f73      ferramentas_jardim   
27039  d1c427060a0f73f6b889a5c7c61f2ac4  informatica_acessorios   
10867  53b36df67ebb7c41585e8d54d6772e08      relogios_presentes   
2794   154e7e31ebfa092203795c972e5804a6            beleza_saude   
8051   3dd2a17168ec895c781a9191c1e95ad7  informatica_acessorios   

       purchase_count  
22112             527  
19742             488  
8613              484  
7364              392  
7079              388  
10840             373  
27039             343  
10867             323  
2794          

In [9]:
from scipy.sparse import csr_matrix

# Create numerical IDs
customer_codes = pd.Categorical(
    purchase_data["customer_id"]
)

product_codes = pd.Categorical(
    purchase_data["product_id"]
)

customer_indices = customer_codes.codes
product_indices = product_codes.codes

interaction_sparse = csr_matrix(
    (
        purchase_data["purchase_count"].astype(float),
        (customer_indices, product_indices)
    ),
    shape=(
        len(customer_codes.categories),
        len(product_codes.categories)
    )
)

print("Sparse interaction matrix shape:", interaction_sparse.shape)
print("Non-zero interactions:", interaction_sparse.nnz)
print("Matrix type:", type(interaction_sparse))

Sparse interaction matrix shape: (98666, 32951)
Non-zero interactions: 102425
Matrix type: <class 'scipy.sparse._csr.csr_matrix'>


In [12]:
# Build a Content-Based Recommendation Baseline

category_summary = (
    products
    .groupby("product_category_name")["product_id"]
    .nunique()
    .sort_values(ascending = False)
)
print("Number of categories:", category_summary.shape[0])
print("\nTop 15 categories by number of products:")
print(category_summary.head(15))


Number of categories: 74

Top 15 categories by number of products:
product_category_name
cama_mesa_banho                3029
esporte_lazer                  2867
moveis_decoracao               2657
beleza_saude                   2444
utilidades_domesticas          2335
automotivo                     1900
informatica_acessorios         1639
brinquedos                     1411
relogios_presentes             1329
telefonia                      1134
bebes                           919
perfumaria                      868
fashion_bolsas_e_acessorios     849
papelaria                       849
cool_stuff                      789
Name: product_id, dtype: int64


In [15]:
# Top popular products
popular_product_ids = (
    purchase_data
    .groupby("product_id")["purchase_count"]
    .sum()
    .sort_values(ascending=False)
    .index
    .tolist()
)

# Products already purchased by each customer
customer_purchased = (
    purchase_data
    .groupby("customer_id")["product_id"]
    .apply(set)
    .to_dict()
)

def recommend_popular(customer_id, n=5):
    
    purchased = customer_purchased.get(customer_id, set())
    
    recommendations = [
        product_id
        for product_id in popular_product_ids
        if product_id not in purchased
    ]
    
    return recommendations[:n]

In [17]:
test_customer = purchase_data["customer_id"].iloc[0]

recommendations = recommend_popular(
    
        test_customer,
        n=5
    )

print("Customer:", test_customer)
print("\nRecommended products:")
print(recommendations)

Customer: 00012a2ce6f8dcda20d059ce98491703

Recommended products:
['aca2eb7d00ea1a7b8ebd4e68314663af', '99a4788cb24856965c36a24e339b6058', '422879e10f46682990de24d770e7f83d', '389d119b48cf3043d311335e499d9c6b', '368c6c730842d78016ad823897a372db']


In [18]:
# Build Item-Based Collaborative Filtering

from sklearn.neighbors import NearestNeighbors


# Item-user Matrix

item_user_matrix = interaction_sparse.T


print("Item-user matrix shape:", item_user_matrix.shape)

# Build nearest - neighbour model

item_model = NearestNeighbors(
  metric = "cosine",
  algorithm = "brute",
   n_neighbors = 11
)
item_model.fit(item_user_matrix)

print("Item-based collaborative filtering model created successfully")


Item-user matrix shape: (32951, 98666)
Item-based collaborative filtering model created successfully


In [20]:
#Personalized Recommendation Function

def recommend_personalized(customer_id, n=5):
    
    purchased_products = customer_purchased.get(customer_id, set())
    
    scores = {}
    
    for product_id in purchased_products:
        
        if product_id not in product_codes.categories:
            continue
        
        product_index = product_codes.categories.get_loc(product_id)
        
        distances, indices = item_model.kneighbors(
            item_user_matrix[product_index],
            n_neighbors=11
        )
        
        for distance, similar_index in zip(distances[0], indices[0]):
            
            similar_product = product_codes.categories[similar_index]
            
            # Skip already purchased product
            if similar_product in purchased_products:
                continue
            
            similarity = 1 - distance
            
            scores[similar_product] = (
                scores.get(similar_product, 0) + similarity
            )
    
    recommendations = sorted(
        scores,
        key=scores.get,
        reverse=True
    )
    
    return recommendations[:n]





In [21]:
test_customer = purchase_data["customer_id"].iloc[0]

personalized_recommendations = recommend_personalized(
    test_customer,
    n=5
)

print("Customer:", test_customer)
print("\nPersonalized recommendations:")
print(personalized_recommendations)

Customer: 00012a2ce6f8dcda20d059ce98491703

Personalized recommendations:
['7cee15aa6f33557c7056ba1fee542b79', '8ef18e8943b9f07649b045b72df449cb', 'fff1059cd247279f3726b7696c66e44e', 'fff0a542c3c62682f23305214eaeaa24', 'ffef256879dbadcab7e77950f4f4a195']


In [22]:
# Make the recommendations understandable
recommendation_details = pd.DataFrame({
    "product_id": personalized_recommendations
})

recommendation_details = recommendation_details.merge(
    product_info,
    on="product_id",
    how="left"
)

print(recommendation_details)

                         product_id product_category_name
0  7cee15aa6f33557c7056ba1fee542b79            brinquedos
1  8ef18e8943b9f07649b045b72df449cb            brinquedos
2  fff1059cd247279f3726b7696c66e44e         esporte_lazer
3  fff0a542c3c62682f23305214eaeaa24             papelaria
4  ffef256879dbadcab7e77950f4f4a195      artigos_de_natal


In [23]:
# Create a simple evaluation dataset

# Sort orders by purchase date
orders["order_purchase_timestamp"] = pd.to_datetime(
    orders["order_purchase_timestamp"]
)

orders_sorted = orders.sort_values(
    ["customer_id", "order_purchase_timestamp"]
)

# Last purchased product for each customer
last_orders = (
    orders_sorted
    .groupby("customer_id")
    .tail(1)
)

test_interactions = last_orders[
    ["customer_id", "order_id"]
].merge(
    order_items[["order_id", "product_id"]],
    on="order_id",
    how="inner"
)

print("Evaluation interactions:", test_interactions.shape)
print(test_interactions.head())

Evaluation interactions: (112650, 3)
                        customer_id                          order_id  \
0  00012a2ce6f8dcda20d059ce98491703  5f79b5b0931d63f1a42989eb65b9da6e   
1  000161a058600d5901f007fab4c27140  a44895d095d7e0702b6a162fa2dbeced   
2  0001fd6190edaaf884bcaf3d49edf079  316a104623542e4d75189bb372bc5f8d   
3  0002414f95344307404f0ace7a26f1d5  5825ce2e88d5346438686b0bba99e5ee   
4  000379cdec625522490c315e70c7a9fb  0ab7fb08086d4af9141453c91878ed7a   

                         product_id  
0  64315bd8c0c47303179dd2e25b579d00  
1  84183944dc7cddca87a5d384452c1d3c  
2  9df2b21ec85378d71df4404712e17478  
3  af3ec22cce878225aae6d9eb6c7a78eb  
4  868b3136c5b206f91b8208fbfdf2cb7c  


In [24]:
# Identify each customer's last order
last_order_ids = (
    orders_sorted
    .groupby("customer_id")["order_id"]
    .last()
)

# Products purchased in the last order = test items
test_interactions = order_items[
    order_items["order_id"].isin(last_order_ids.values)
].merge(
    orders_sorted[["order_id", "customer_id"]],
    on="order_id",
    how="inner"
)[["customer_id", "product_id"]].drop_duplicates()

# Remove the last-order products from customer history
train_interactions = customer_product_unique.merge(
    test_interactions,
    on=["customer_id", "product_id"],
    how="left",
    indicator=True
)

train_interactions = train_interactions[
    train_interactions["_merge"] == "left_only"
].drop(columns="_merge")

print("Training interactions:", train_interactions.shape)
print("Test interactions:", test_interactions.shape)

print("\nUnique test customers:",
      test_interactions["customer_id"].nunique())

print("\nSample test interactions:")
print(test_interactions.head())

Training interactions: (0, 2)
Test interactions: (102425, 2)

Unique test customers: 98666

Sample test interactions:
                        customer_id                        product_id
0  3ce436f183e68e07877b285a838db11a  4244733e06e7ecb4970a6e2683c13e61
1  f6dd3ec061db4e3987629fe6b26e5cce  e5f2d52b802189ee658865ca93d83a8f
2  6489ae5e4333f3693df5ad4372dab6d3  c777355d18b72b67abbeef9df44fd0fd
3  d4eb9395c8c0431ee92fce09860c5a06  7634da152a4610f1595efa32f14722fc
4  58dbd0b2d70206bf40e62cd34e84d795  ac6c3623068f30de03045865e4e10089


In [25]:
# Correct evaluation setup
# Add purchase timestamp to order items
evaluation_data = order_items.merge(
    orders_sorted[
        ["order_id", "customer_id", "order_purchase_timestamp"]
    ],
    on="order_id",
    how="inner"
)

# Last purchase date for each customer
last_purchase = (
    evaluation_data
    .groupby("customer_id")["order_purchase_timestamp"]
    .max()
    .reset_index(name="last_purchase_timestamp")
)

# Test set = products from the customer's last order
test_interactions = evaluation_data.merge(
    last_purchase,
    on="customer_id",
    how="inner"
)

test_interactions = test_interactions[
    test_interactions["order_purchase_timestamp"]
    == test_interactions["last_purchase_timestamp"]
][
    ["customer_id", "product_id"]
].drop_duplicates()

# Training history = products purchased BEFORE the last purchase
train_interactions = evaluation_data.merge(
    last_purchase,
    on="customer_id",
    how="inner"
)

train_interactions = train_interactions[
    train_interactions["order_purchase_timestamp"]
    < train_interactions["last_purchase_timestamp"]
][
    ["customer_id", "product_id"]
].drop_duplicates()

print("Training interactions:", train_interactions.shape)
print("Test interactions:", test_interactions.shape)

print(
    "Customers with training history:",
    train_interactions["customer_id"].nunique()
)

print(
    "Customers in test set:",
    test_interactions["customer_id"].nunique()
)

print("\nSample training interactions:")
print(train_interactions.head())

print("\nSample test interactions:")
print(test_interactions.head())

Training interactions: (0, 2)
Test interactions: (102425, 2)
Customers with training history: 0
Customers in test set: 98666

Sample training interactions:
Empty DataFrame
Columns: [customer_id, product_id]
Index: []

Sample test interactions:
                        customer_id                        product_id
0  3ce436f183e68e07877b285a838db11a  4244733e06e7ecb4970a6e2683c13e61
1  f6dd3ec061db4e3987629fe6b26e5cce  e5f2d52b802189ee658865ca93d83a8f
2  6489ae5e4333f3693df5ad4372dab6d3  c777355d18b72b67abbeef9df44fd0fd
3  d4eb9395c8c0431ee92fce09860c5a06  7634da152a4610f1595efa32f14722fc
4  58dbd0b2d70206bf40e62cd34e84d795  ac6c3623068f30de03045865e4e10089


In [26]:
# Evaluation customers with ≥2 orders
# Count orders per customer
customer_order_counts = (
    orders_sorted
    .groupby("customer_id")["order_id"]
    .nunique()
)

# Keep only customers with at least 2 orders
repeat_customers = customer_order_counts[
    customer_order_counts >= 2
].index

print("Customers with at least 2 orders:",
      len(repeat_customers))

# Keep only these customers
evaluation_data = evaluation_data[
    evaluation_data["customer_id"].isin(repeat_customers)
].copy()

# Find each customer's last purchase timestamp
last_purchase = (
    evaluation_data
    .groupby("customer_id")["order_purchase_timestamp"]
    .max()
    .reset_index(name="last_purchase_timestamp")
)

# Test = products from final order
test_interactions = evaluation_data.merge(
    last_purchase,
    on="customer_id",
    how="inner"
)

test_interactions = test_interactions[
    test_interactions["order_purchase_timestamp"]
    == test_interactions["last_purchase_timestamp"]
][
    ["customer_id", "product_id"]
].drop_duplicates()

# Training = products bought before final order
train_interactions = evaluation_data.merge(
    last_purchase,
    on="customer_id",
    how="inner"
)

train_interactions = train_interactions[
    train_interactions["order_purchase_timestamp"]
    < train_interactions["last_purchase_timestamp"]
][
    ["customer_id", "product_id"]
].drop_duplicates()

print("\nTraining interactions:", train_interactions.shape)
print("Test interactions:", test_interactions.shape)

print(
    "Training customers:",
    train_interactions["customer_id"].nunique()
)

print(
    "Test customers:",
    test_interactions["customer_id"].nunique()
)

Customers with at least 2 orders: 0

Training interactions: (0, 2)
Test interactions: (0, 2)
Training customers: 0
Test customers: 0


In [27]:
# Recreate orders_sorted from the original orders dataframe
orders_eval = orders.copy()

orders_eval["order_purchase_timestamp"] = pd.to_datetime(
    orders_eval["order_purchase_timestamp"]
)

# Count orders per customer
customer_order_counts = (
    orders_eval
    .groupby("customer_id")["order_id"]
    .nunique()
)

print("Total customers:", customer_order_counts.shape[0])

print(
    "Customers with 2+ orders:",
    (customer_order_counts >= 2).sum()
)

print("\nOrder count distribution:")
print(
    customer_order_counts.value_counts()
    .sort_index()
    .head(10)
)

Total customers: 99441
Customers with 2+ orders: 0

Order count distribution:
order_id
1    99441
Name: count, dtype: int64


In [28]:
# Check orders and customer mapping
print("Orders columns:")
print(orders.columns.tolist())

print("\nCustomers columns:")
print(pd.read_csv("../data/processed/customers_clean.csv").columns.tolist())

Orders columns:
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'delivery_days', 'estimated_delivery_days', 'delivery_delay_days', 'delivery_performance']

Customers columns:
['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']


In [29]:
# Load customer mapping
customers = pd.read_csv(
    "../data/processed/customers_clean.csv"
)

# Add unique customer ID to orders
orders_eval = orders.merge(
    customers[
        ["customer_id", "customer_unique_id"]
    ],
    on="customer_id",
    how="left"
)

print("Orders shape:", orders_eval.shape)

print("\nMissing unique customer IDs:",
      orders_eval["customer_unique_id"].isna().sum())

# Count orders per real customer
customer_order_counts = (
    orders_eval
    .groupby("customer_unique_id")["order_id"]
    .nunique()
)

print(
    "\nUnique customers:",
    customer_order_counts.shape[0]
)

print(
    "Customers with 2+ orders:",
    (customer_order_counts >= 2).sum()
)

print("\nOrder count distribution:")
print(
    customer_order_counts
    .value_counts()
    .sort_index()
    .head(10)
)

Orders shape: (99441, 13)

Missing unique customer IDs: 0

Unique customers: 96096
Customers with 2+ orders: 2997

Order count distribution:
order_id
1     93099
2      2745
3       203
4        30
5         8
6         6
7         3
9         1
17        1
Name: count, dtype: int64


In [30]:
# Sort orders by customer and purchase time
orders_eval = orders_eval.sort_values(
    ["customer_unique_id", "order_purchase_timestamp"]
)

# Get the last order for each repeat customer
repeat_customers = customer_order_counts[
    customer_order_counts >= 2
].index

repeat_orders = orders_eval[
    orders_eval["customer_unique_id"].isin(repeat_customers)
].copy()

last_orders = (
    repeat_orders
    .groupby("customer_unique_id")
    .tail(1)
)

# Test = products purchased in the last order
test_interactions = last_orders[
    ["customer_unique_id", "order_id"]
].merge(
    order_items[
        ["order_id", "product_id"]
    ],
    on="order_id",
    how="inner"
)[
    ["customer_unique_id", "product_id"]
].drop_duplicates()

# Training = products purchased before the last order
last_order_ids = set(last_orders["order_id"])

train_orders = repeat_orders[
    ~repeat_orders["order_id"].isin(last_order_ids)
]

train_interactions = train_orders[
    ["customer_unique_id", "order_id"]
].merge(
    order_items[
        ["order_id", "product_id"]
    ],
    on="order_id",
    how="inner"
)[
    ["customer_unique_id", "product_id"]
].drop_duplicates()

print("Training interactions:", train_interactions.shape)
print("Test interactions:", test_interactions.shape)

print(
    "\nTraining customers:",
    train_interactions["customer_unique_id"].nunique()
)

print(
    "Test customers:",
    test_interactions["customer_unique_id"].nunique()
)

print("\nSample training data:")
print(train_interactions.head())

print("\nSample test data:")
print(test_interactions.head())

Training interactions: (3485, 2)
Test interactions: (3148, 2)

Training customers: 2959
Test customers: 2943

Sample training data:
                 customer_unique_id                        product_id
0  00172711b30d52eea8b313a7f2cced02  8f71fe1e1b5bbb6773e7276b5c37ef65
1  004288347e5e88a27ded2bb23747066c  6e1b14d3cbb5fb3a2c00351007127dfd
2  004b45ec5c64187465168251cd1c9c2f  b0961721fd839e9982420e807758a2a6
3  0058f300f57d7b93c477a131a59b36c3  6bd248f93425ceeb625a8a97e2404112
4  00a39521eb40f7012db50455bf083460  a42d9c825894f96fc6ed02610891454d

Sample test data:
                 customer_unique_id                        product_id
0  004288347e5e88a27ded2bb23747066c  a2bd2eae20998a24c22b110334928b02
1  004b45ec5c64187465168251cd1c9c2f  a6ad77b15e566298a4e8ee2011ab1255
2  0058f300f57d7b93c477a131a59b36c3  4630761de87581e8b659dc77bb7eb4ee
3  00a39521eb40f7012db50455bf083460  f89cd865cac300a9bf1320dd8f0fa223
4  00cc12a6d8b578b8ebd21ea4e2ae8b27  48ee9be392f28ae3a64518a070f4d06a


In [31]:
# Create customer-product frequency from training history
eval_purchase_frequency = (
    train_interactions
    .groupby(
        ["customer_unique_id", "product_id"]
    )
    .size()
    .reset_index(name="purchase_count")
)

# Encode customers and products
eval_customer_codes = pd.Categorical(
    eval_purchase_frequency["customer_unique_id"]
)

eval_product_codes = pd.Categorical(
    eval_purchase_frequency["product_id"]
)

eval_customer_indices = eval_customer_codes.codes
eval_product_indices = eval_product_codes.codes

# Sparse customer-product matrix
eval_interaction_sparse = csr_matrix(
    (
        eval_purchase_frequency["purchase_count"].astype(float),
        (
            eval_customer_indices,
            eval_product_indices
        )
    ),
    shape=(
        len(eval_customer_codes.categories),
        len(eval_product_codes.categories)
    )
)

print(
    "Evaluation sparse matrix shape:",
    eval_interaction_sparse.shape
)

print(
    "Non-zero interactions:",
    eval_interaction_sparse.nnz
)

Evaluation sparse matrix shape: (2959, 2816)
Non-zero interactions: 3485


In [32]:
# Item-user matrix for evaluation
eval_item_user_matrix = eval_interaction_sparse.T

# Number of neighbors
n_neighbors = min(11, eval_item_user_matrix.shape[0])

# Build evaluation model
eval_item_model = NearestNeighbors(
    metric="cosine",
    algorithm="brute",
    n_neighbors=n_neighbors
)

eval_item_model.fit(eval_item_user_matrix)

print(
    "Evaluation item-user matrix shape:",
    eval_item_user_matrix.shape
)

print(
    "Evaluation collaborative filtering model created successfully"
)

Evaluation item-user matrix shape: (2816, 2959)
Evaluation collaborative filtering model created successfully


In [33]:
# Create the evaluation recommendation function
def recommend_eval(customer_id, n=5):
    
    # Products purchased in training history
    customer_rows = eval_purchase_frequency[
        eval_purchase_frequency["customer_unique_id"] == customer_id
    ]
    
    purchased_products = set(
        customer_rows["product_id"]
    )
    
    scores = {}
    
    for product_id in purchased_products:
        
        # Product must exist in evaluation product mapping
        if product_id not in eval_product_codes.categories:
            continue
        
        product_index = eval_product_codes.categories.get_loc(
            product_id
        )
        
        distances, indices = eval_item_model.kneighbors(
            eval_item_user_matrix[product_index],
            n_neighbors=n_neighbors
        )
        
        for distance, similar_index in zip(
            distances[0],
            indices[0]
        ):
            
            similar_product = eval_product_codes.categories[
                similar_index
            ]
            
            # Don't recommend already purchased products
            if similar_product in purchased_products:
                continue
            
            similarity = 1 - distance
            
            scores[similar_product] = (
                scores.get(similar_product, 0)
                + similarity
            )
    
    recommendations = sorted(
        scores,
        key=scores.get,
        reverse=True
    )
    
    return recommendations[:n]

In [34]:
test_customer = train_interactions[
    "customer_unique_id"
].iloc[0]

eval_recommendations = recommend_eval(
    test_customer,
    n=5
)

print("Customer:", test_customer)
print("\nEvaluation recommendations:")
print(eval_recommendations)

Customer: 00172711b30d52eea8b313a7f2cced02

Evaluation recommendations:
['ffd4bf4306745865e5692f69bd237893', 'ffc0b406806006602c5853b00ab5f7fd', 'ffbbf6b9097237a1122f17e7341a3fb2', 'ffaaddefb271481c66d4bd79844ecdae', 'ffa7e0cbe11656d11a117b534bb1db27']


In [35]:
# valuate the recommender
precisions = []
recalls = []
hits = []

# Evaluate customers who have both training and test history
evaluation_customers = sorted(
    set(train_interactions["customer_unique_id"])
    & set(test_interactions["customer_unique_id"])
)

for customer_id in evaluation_customers:
    
    # Actual products purchased in the final order
    actual_products = set(
        test_interactions[
            test_interactions["customer_unique_id"] == customer_id
        ]["product_id"]
    )
    
    # Recommended products
    recommended_products = set(
        recommend_eval(customer_id, n=5)
    )
    
    if len(recommended_products) == 0:
        continue
    
    true_positives = len(
        actual_products & recommended_products
    )
    
    precision = true_positives / len(recommended_products)
    recall = (
        true_positives / len(actual_products)
        if len(actual_products) > 0
        else 0
    )
    
    precisions.append(precision)
    recalls.append(recall)
    hits.append(1 if true_positives > 0 else 0)

precision_at_5 = np.mean(precisions)
recall_at_5 = np.mean(recalls)
hit_rate_at_5 = np.mean(hits)

print("Evaluation customers:", len(evaluation_customers))
print(f"Precision@5: {precision_at_5:.4f}")
print(f"Recall@5:    {recall_at_5:.4f}")
print(f"Hit Rate@5:  {hit_rate_at_5:.4f}")


Evaluation customers: 2913
Precision@5: 0.0035
Recall@5:    0.0173
Hit Rate@5:  0.0175


In [36]:
recommendation_evaluation = pd.DataFrame({
    "metric": [
        "Precision@5",
        "Recall@5",
        "Hit Rate@5"
    ],
    "value": [
        precision_at_5,
        recall_at_5,
        hit_rate_at_5
    ]
})

recommendation_evaluation.to_csv(
    "../data/processed/recommendation_evaluation.csv",
    index=False
)

print(recommendation_evaluation)
print("\nSaved: recommendation_evaluation.csv")

        metric     value
0  Precision@5  0.003504
1     Recall@5  0.017348
2   Hit Rate@5  0.017520

Saved: recommendation_evaluation.csv


In [37]:
# Create recommendation results for evaluation customers

recommendation_rows = []

for customer_id in evaluation_customers[:100]:
    
    recommendations = recommend_eval(
        customer_id,
        n=5
    )
    
    for rank, product_id in enumerate(
        recommendations,
        start=1
    ):
        recommendation_rows.append({
            "customer_unique_id": customer_id,
            "rank": rank,
            "recommended_product_id": product_id
        })

recommendation_results = pd.DataFrame(
    recommendation_rows
)

recommendation_results.to_csv(
    "../data/processed/recommendation_results.csv",
    index=False
)

print("Recommendation results shape:",
      recommendation_results.shape)

print("\nSample:")
print(recommendation_results.head(10))

print("\nSaved: recommendation_results.csv")

Recommendation results shape: (500, 3)

Sample:
                 customer_unique_id  rank            recommended_product_id
0  004288347e5e88a27ded2bb23747066c     1  ffd4bf4306745865e5692f69bd237893
1  004288347e5e88a27ded2bb23747066c     2  ffc0b406806006602c5853b00ab5f7fd
2  004288347e5e88a27ded2bb23747066c     3  ffbbf6b9097237a1122f17e7341a3fb2
3  004288347e5e88a27ded2bb23747066c     4  ffaaddefb271481c66d4bd79844ecdae
4  004288347e5e88a27ded2bb23747066c     5  ffa7e0cbe11656d11a117b534bb1db27
5  004b45ec5c64187465168251cd1c9c2f     1  ffd4bf4306745865e5692f69bd237893
6  004b45ec5c64187465168251cd1c9c2f     2  ffc0b406806006602c5853b00ab5f7fd
7  004b45ec5c64187465168251cd1c9c2f     3  ffbbf6b9097237a1122f17e7341a3fb2
8  004b45ec5c64187465168251cd1c9c2f     4  ffaaddefb271481c66d4bd79844ecdae
9  004b45ec5c64187465168251cd1c9c2f     5  ffa7e0cbe11656d11a117b534bb1db27

Saved: recommendation_results.csv


In [38]:
# Improve personalization
unique_recommended_products = (
    recommendation_results["recommended_product_id"]
    .nunique()
)

total_recommendations = len(
    recommendation_results
)

print("Total recommendations:",
      total_recommendations)

print("Unique recommended products:",
      unique_recommended_products)

print(
    "Recommendation diversity:",
    f"{unique_recommended_products / total_recommendations:.2%}"
)

Total recommendations: 500
Unique recommended products: 27
Recommendation diversity: 5.40%


In [39]:
# Improve personalized scoring
def recommend_eval_weighted(customer_id, n=5):
    
    # Customer's training purchase history
    customer_rows = eval_purchase_frequency[
        eval_purchase_frequency["customer_unique_id"] == customer_id
    ]
    
    purchased_products = set(
        customer_rows["product_id"]
    )
    
    scores = {}
    
    for _, row in customer_rows.iterrows():
        
        product_id = row["product_id"]
        purchase_count = row["purchase_count"]
        
        if product_id not in eval_product_codes.categories:
            continue
        
        product_index = eval_product_codes.categories.get_loc(
            product_id
        )
        
        distances, indices = eval_item_model.kneighbors(
            eval_item_user_matrix[product_index],
            n_neighbors=n_neighbors
        )
        
        for distance, similar_index in zip(
            distances[0],
            indices[0]
        ):
            
            similar_product = eval_product_codes.categories[
                similar_index
            ]
            
            if similar_product in purchased_products:
                continue
            
            similarity = 1 - distance
            
            # Weight similarity by purchase frequency
            score = similarity * purchase_count
            
            scores[similar_product] = (
                scores.get(similar_product, 0) + score
            )
    
    recommendations = sorted(
        scores,
        key=scores.get,
        reverse=True
    )
    
    return recommendations[:n]

In [40]:
test_customer = evaluation_customers[0]

weighted_recommendations = recommend_eval_weighted(
    test_customer,
    n=5
)

print("Customer:", test_customer)
print("\nWeighted personalized recommendations:")
print(weighted_recommendations)

Customer: 004288347e5e88a27ded2bb23747066c

Weighted personalized recommendations:
['ffd4bf4306745865e5692f69bd237893', 'ffc0b406806006602c5853b00ab5f7fd', 'ffbbf6b9097237a1122f17e7341a3fb2', 'ffaaddefb271481c66d4bd79844ecdae', 'ffa7e0cbe11656d11a117b534bb1db27']


In [41]:
weighted_precisions = []
weighted_recalls = []
weighted_hits = []

for customer_id in evaluation_customers:
    
    actual_products = set(
        test_interactions[
            test_interactions["customer_unique_id"] == customer_id
        ]["product_id"]
    )
    
    recommended_products = set(
        recommend_eval_weighted(
            customer_id,
            n=5
        )
    )
    
    if len(recommended_products) == 0:
        continue
    
    true_positives = len(
        actual_products & recommended_products
    )
    
    precision = true_positives / len(recommended_products)
    
    recall = (
        true_positives / len(actual_products)
        if len(actual_products) > 0
        else 0
    )
    
    weighted_precisions.append(precision)
    weighted_recalls.append(recall)
    weighted_hits.append(
        1 if true_positives > 0 else 0
    )

weighted_precision_at_5 = np.mean(
    weighted_precisions
)

weighted_recall_at_5 = np.mean(
    weighted_recalls
)

weighted_hit_rate_at_5 = np.mean(
    weighted_hits
)

print("Evaluation customers:",
      len(evaluation_customers))

print(
    f"Weighted Precision@5: {weighted_precision_at_5:.4f}"
)

print(
    f"Weighted Recall@5:    {weighted_recall_at_5:.4f}"
)

print(
    f"Weighted Hit Rate@5:  {weighted_hit_rate_at_5:.4f}"
)

Evaluation customers: 2913
Weighted Precision@5: 0.0035
Weighted Recall@5:    0.0173
Weighted Hit Rate@5:  0.0175


In [42]:
weighted_evaluation = pd.DataFrame({
    "model": ["Item-Based CF Weighted"],
    "precision_at_5": [weighted_precision_at_5],
    "recall_at_5": [weighted_recall_at_5],
    "hit_rate_at_5": [weighted_hit_rate_at_5]
})

weighted_evaluation.to_csv(
    "../data/processed/recommendation_weighted_evaluation.csv",
    index=False
)

print(weighted_evaluation)
print("\nWeighted evaluation saved successfully.")

                    model  precision_at_5  recall_at_5  hit_rate_at_5
0  Item-Based CF Weighted        0.003504     0.017348        0.01752

Weighted evaluation saved successfully.


In [43]:
# Compare both recommendation models
comparison = pd.DataFrame({
    "model": [
        "Item-Based CF",
        "Item-Based CF Weighted"
    ],
    "precision_at_5": [
        0.0035,
        weighted_precision_at_5
    ],
    "recall_at_5": [
        0.0173,
        weighted_recall_at_5
    ],
    "hit_rate_at_5": [
        0.0175,
        weighted_hit_rate_at_5
    ]
})

print(comparison)


                    model  precision_at_5  recall_at_5  hit_rate_at_5
0           Item-Based CF        0.003500     0.017300        0.01750
1  Item-Based CF Weighted        0.003504     0.017348        0.01752


In [44]:
# Final recommendation strategy

def recommend_final(customer_id, n=5):
    # Try personalized recommendations first
    recommendations = recommend_eval(customer_id, n=n)

    # If enough personalized recommendations are available
    if len(recommendations) >= n:
        return recommendations[:n]

    # Get products already purchased
    customer_rows = eval_purchase_frequency[
        eval_purchase_frequency["customer_unique_id"] == customer_id
    ]

    purchased_products = set(customer_rows["product_id"])

    # Popular products from training data
    popular_eval_products = (
        train_interactions
        .groupby("product_id")
        .size()
        .sort_values(ascending=False)
        .index.tolist()
    )

    # Add popular products as fallback
    for product_id in popular_eval_products:
        if product_id not in purchased_products and product_id not in recommendations:
            recommendations.append(product_id)

        if len(recommendations) == n:
            break

    return recommendations


In [45]:
test_customer = evaluation_customers[0]

final_recommendations = recommend_final(
    test_customer,
    n=5
)

print("Customer:", test_customer)
print("Final Recommendations:")

for rank, product_id in enumerate(final_recommendations, start=1):
    print(rank, product_id)

Customer: 004288347e5e88a27ded2bb23747066c
Final Recommendations:
1 ffd4bf4306745865e5692f69bd237893
2 ffc0b406806006602c5853b00ab5f7fd
3 ffbbf6b9097237a1122f17e7341a3fb2
4 ffaaddefb271481c66d4bd79844ecdae
5 ffa7e0cbe11656d11a117b534bb1db27


In [46]:
# Add product categories
final_recommendation_details = pd.DataFrame({
    "rank": range(1, len(final_recommendations) + 1),
    "product_id": final_recommendations
}).merge(
    product_info,
    on="product_id",
    how="left"
)

print(final_recommendation_details)

   rank                        product_id        product_category_name
0     1  ffd4bf4306745865e5692f69bd237893  fashion_bolsas_e_acessorios
1     2  ffc0b406806006602c5853b00ab5f7fd             artigos_de_natal
2     3  ffbbf6b9097237a1122f17e7341a3fb2                esporte_lazer
3     4  ffaaddefb271481c66d4bd79844ecdae        utilidades_domesticas
4     5  ffa7e0cbe11656d11a117b534bb1db27                 beleza_saude


In [47]:
# Generate final recommendations for 100 customers
final_recommendation_rows = []

for customer_id in evaluation_customers[:100]:

    recommendations = recommend_final(
        customer_id,
        n=5
    )

    for rank, product_id in enumerate(recommendations, start=1):

        category = product_info.loc[
            product_info["product_id"] == product_id,
            "product_category_name"
        ]

        category = category.iloc[0] if len(category) > 0 else "Unknown"

        final_recommendation_rows.append({
            "customer_unique_id": customer_id,
            "rank": rank,
            "recommended_product_id": product_id,
            "product_category_name": category
        })

final_recommendation_results = pd.DataFrame(
    final_recommendation_rows
)

print("Shape:", final_recommendation_results.shape)
print(final_recommendation_results.head(10))

Shape: (500, 4)
                 customer_unique_id  rank            recommended_product_id  \
0  004288347e5e88a27ded2bb23747066c     1  ffd4bf4306745865e5692f69bd237893   
1  004288347e5e88a27ded2bb23747066c     2  ffc0b406806006602c5853b00ab5f7fd   
2  004288347e5e88a27ded2bb23747066c     3  ffbbf6b9097237a1122f17e7341a3fb2   
3  004288347e5e88a27ded2bb23747066c     4  ffaaddefb271481c66d4bd79844ecdae   
4  004288347e5e88a27ded2bb23747066c     5  ffa7e0cbe11656d11a117b534bb1db27   
5  004b45ec5c64187465168251cd1c9c2f     1  ffd4bf4306745865e5692f69bd237893   
6  004b45ec5c64187465168251cd1c9c2f     2  ffc0b406806006602c5853b00ab5f7fd   
7  004b45ec5c64187465168251cd1c9c2f     3  ffbbf6b9097237a1122f17e7341a3fb2   
8  004b45ec5c64187465168251cd1c9c2f     4  ffaaddefb271481c66d4bd79844ecdae   
9  004b45ec5c64187465168251cd1c9c2f     5  ffa7e0cbe11656d11a117b534bb1db27   

         product_category_name  
0  fashion_bolsas_e_acessorios  
1             artigos_de_natal  
2              

In [48]:
final_recommendation_results.to_csv(
    "../data/processed/final_recommendation_results.csv",
    index=False
)

print("Final recommendation results saved successfully.")
print("Shape:", final_recommendation_results.shape)

Final recommendation results saved successfully.
Shape: (500, 4)


In [49]:
# Check recommendation coverage and diversity
print("Total recommendation rows:", len(final_recommendation_results))

print(
    "Unique customers:",
    final_recommendation_results["customer_unique_id"].nunique()
)

print(
    "Unique recommended products:",
    final_recommendation_results["recommended_product_id"].nunique()
)

print(
    "Unique categories:",
    final_recommendation_results["product_category_name"].nunique()
)

print("\nRecommendations per customer:")
print(
    final_recommendation_results
    .groupby("customer_unique_id")
    .size()
    .value_counts()
    .sort_index()
)

Total recommendation rows: 500
Unique customers: 100
Unique recommended products: 27
Unique categories: 13

Recommendations per customer:
5    100
Name: count, dtype: int64
